# OnKith research figures

Every table and plot for the report, built from evidence already in the
repository. This notebook **trains nothing, evaluates nothing and downloads
nothing** - it reads saved metrics and draws them.

Run All. Each section either produces its figure or says exactly which artifact
is missing; a missing input never stops the notebook.

**Outputs**

```text
research_results/
├── figures/      PNG (300 dpi) and PDF for every plot
├── tables/       the numbers behind each figure, as CSV
└── summaries/    figure index and a written overview
```

**Deployment figures.** Section 9 reads whatever
`Full_Pipeline_CPU_Benchmark.ipynb` has left under
`research_results/cpu_benchmarks/`. Before that notebook has been run there is
nothing to draw and the section says so. Once the Raspberry Pi results are
committed and pulled, rerunning this notebook adds the device-comparison and
parity figures with no edits.

**Honesty rule.** A metric is only plotted beside another metric when the two
were measured the same way. Where a historical model has no comparable number,
this notebook says "not recorded" rather than inventing one or quietly
substituting a different metric.

## 0. Setup

In [ ]:
import csv
import json
import math
import statistics
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # figures are files, not windows; keeps Run All headless-safe
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "FullPipeline_V3").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "FullPipeline_V3").is_dir():
    raise RuntimeError("Run this notebook from inside the OnKith repository.")

RESULTS = REPO_ROOT / "research_results"
FIGURES = RESULTS / "figures"
TABLES = RESULTS / "tables"
SUMMARIES = RESULTS / "summaries"
CPU_BENCHMARKS = RESULTS / "cpu_benchmarks"
for directory in (FIGURES, TABLES, SUMMARIES, CPU_BENCHMARKS):
    directory.mkdir(parents=True, exist_ok=True)

# --- publication styling -----------------------------------------------------
# Categorical hues are assigned in a fixed order and never cycled: blue is always
# the current deployment model, orange the model it replaced, aqua the third
# series where one exists.
SERIES = {"primary": "#2a78d6", "secondary": "#eb6834", "tertiary": "#1baf7a"}
INK = "#0b0b0b"
MUTED = "#52514e"
GRID = "#d9d8d4"

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 10.5,
    "axes.titlesize": 12.5,
    "axes.labelsize": 10.5,
    "axes.titleweight": "bold",
    "axes.edgecolor": GRID,
    "axes.labelcolor": INK,
    "axes.titlecolor": INK,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": GRID,
    "grid.linewidth": 0.7,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.frameon": False,
    "legend.fontsize": 9.5,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "lines.linewidth": 2.0,
    "lines.markersize": 6,
})

GENERATED: list[dict] = []      # every figure and table this run produced
UNAVAILABLE: list[str] = []     # every input that was missing, with the reason


def tidy(axis, *, x_grid: bool = False) -> None:
    """Recessive frame: no top/right spines, grid on the value axis only."""

    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.grid(axis="x" if x_grid else "y")
    axis.set_axisbelow(True)


def save(figure, name: str, caption: str) -> None:
    """PNG for the report, PDF for the vector version, both at print quality."""

    png, pdf = FIGURES / f"{name}.png", FIGURES / f"{name}.pdf"
    figure.savefig(png)
    figure.savefig(pdf)
    plt.close(figure)
    GENERATED.append({"kind": "figure", "name": name,
                      "files": [png.name, f"{pdf.name}"], "caption": caption})
    print(f"  figure  {png.relative_to(REPO_ROOT).as_posix()}  (+ .pdf)")


def save_table(rows, fieldnames, name: str, caption: str) -> None:
    if not rows:
        return
    path = TABLES / f"{name}.csv"
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames, lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)
    GENERATED.append({"kind": "table", "name": name, "files": [path.name],
                      "caption": caption, "rows": len(rows)})
    print(f"  table   {path.relative_to(REPO_ROOT).as_posix()}  ({len(rows)} rows)")


def load_json(path: Path, label: str):
    if not Path(path).is_file():
        UNAVAILABLE.append(f"{label}: {Path(path).relative_to(REPO_ROOT).as_posix()} not found")
        return None
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        UNAVAILABLE.append(f"{label}: unreadable ({exc})")
        return None


print("repository :", REPO_ROOT)
print("figures    :", FIGURES.relative_to(REPO_ROOT).as_posix())
print("tables     :", TABLES.relative_to(REPO_ROOT).as_posix())

## 1. Discover the evidence

Everything this notebook can draw, and whether it is present.

In [ ]:
MODEL_V2 = REPO_ROOT / "model" / "DeBERTa-v3-xsmall"
METRICS = MODEL_V2 / "artifacts" / "model_v2" / "metrics"
EVALUATION = REPO_ROOT / "model" / "privacy_evaluation"

DEBERTA_FP32 = load_json(METRICS / "evaluation_production_path_fp32_onnx.json", "DeBERTa FP32 ONNX")
DEBERTA_INT8 = load_json(METRICS / "evaluation_production_path_int8_onnx.json", "DeBERTa INT8 ONNX")
ACCEPTED = load_json(METRICS / "accepted_int8_candidate.json", "accepted INT8 candidate")
OOD_FINAL_DEBERTA = load_json(METRICS / "evaluation_ood_final_int8.json", "DeBERTa frozen OOD final")
DEPLOY_MANIFEST = load_json(
    REPO_ROOT / "FullPipeline_V3" / "models" / "deberta_pii" / "model_manifest.json",
    "deployment manifest",
)

TINYBERT_IID = load_json(EVALUATION / "baselines" / "tinybert_iid_baseline.json", "TinyBERT IID baseline")
TINYBERT_OOD_DEV = load_json(EVALUATION / "results" / "tinybert_dev_summary.json", "TinyBERT OOD dev")
TINYBERT_OOD_FINAL = load_json(EVALUATION / "results" / "tinybert_final_summary.json", "TinyBERT OOD final")

BILSTM = load_json(REPO_ROOT / "model" / "artifacts" / "final_eval_metrics.json", "BiLSTM baseline")
PREFLIGHT = load_json(
    MODEL_V2 / "artifacts" / "preflight" / "metrics" / "final_benchmark.json",
    "workstation preflight benchmark",
)

VIEW_ORDER = ["test", "val_original", "val_supplementary", "combined_validation", "ood_dev"]
VIEW_LABELS = {
    "test": "Test",
    "val_original": "Validation\n(original)",
    "val_supplementary": "Validation\n(supplementary)",
    "combined_validation": "Combined\nvalidation",
    "ood_dev": "OOD dev",
    "ood_final": "OOD final\n(frozen)",
}
# Short forms for the axes that carry six groups; the descriptive names above
# stay available for captions and prose.
VIEW_TICKS = {
    "test": "Test",
    "val_original": "Val\n(orig)",
    "val_supplementary": "Val\n(supp)",
    "combined_validation": "Combined\nval",
    "ood_dev": "OOD\ndev",
    "ood_final": "OOD final\n(frozen)",
}

sources = [
    ("DeBERTa INT8, five evaluation views", DEBERTA_INT8),
    ("DeBERTa FP32, five evaluation views", DEBERTA_FP32),
    ("DeBERTa frozen OOD final (one-shot)", OOD_FINAL_DEBERTA),
    ("Accepted INT8 candidate / quality gate", ACCEPTED),
    ("TinyBERT IID baseline (historical)", TINYBERT_IID),
    ("TinyBERT OOD dev", TINYBERT_OOD_DEV),
    ("TinyBERT OOD final", TINYBERT_OOD_FINAL),
    ("BiLSTM baseline", BILSTM),
    ("Workstation latency preflight", PREFLIGHT),
]
print(f"{'source':44} available")
print("-" * 56)
for label, payload in sources:
    print(f"{label:44} {'yes' if payload else 'NO'}")

benchmark_runs = sorted(
    directory for directory in CPU_BENCHMARKS.glob("*")
    if directory.is_dir() and (directory / "benchmark_config.json").is_file()
)
print(f"\nCPU benchmark runs found: {len(benchmark_runs)}")
for directory in benchmark_runs:
    print(f"  {directory.name}")

## 2. Model progression

Three generations of privacy model. Only numbers measured the same way are put
on the same axis:

* **BiLSTM** was scored with token-level *binary* accuracy on its own validation
  split. There is no span-level evaluation for it, so it has no bar here. Its
  recorded number is in the table, flagged as not comparable.
* **TinyBERT-4 INT8** and **DeBERTa-v3-xsmall INT8** were scored on the *same*
  40,909-row held-out AI4Privacy English validation split, through the same
  metric definitions, in `deberta_vs_tinybert_int8_validation.ipynb`. Those two
  are directly comparable and are what the figure shows.

In [ ]:
progression_rows = []
head_to_head = (DEPLOY_MANIFEST or {}).get("held_out_validation")

if BILSTM:
    progression_rows.append({
        "model": "BiLSTM baseline",
        "generation": "v0",
        "precision": "fp32",
        "metric": "token-level binary F1",
        "value": BILSTM["val_f1"],
        "precision_value": BILSTM["val_precision"],
        "recall_value": BILSTM["val_recall"],
        "evaluation": "own validation split",
        "comparable_to_span_metrics": False,
    })

if head_to_head:
    for key, name, generation in (
        ("tinybert_int8_model_v1", "TinyBERT-4 INT8", "v1"),
        ("deberta_int8", "DeBERTa-v3-xsmall INT8", "v2"),
    ):
        entry = head_to_head[key]
        progression_rows.append({
            "model": name,
            "generation": generation,
            "precision": "int8",
            "metric": "typed entity F1",
            "value": entry["typed_f1"],
            "precision_value": None,
            "recall_value": None,
            "evaluation": f"shared held-out validation, {head_to_head['rows']:,} rows",
            "comparable_to_span_metrics": True,
            "private_character_leakage": entry["private_character_leakage"],
            "non_private_overmasking_rate": entry["non_private_overmasking_rate"],
        })

if head_to_head:
    comparable = [row for row in progression_rows if row["comparable_to_span_metrics"]]
    names = [row["model"] for row in comparable]
    f1 = [row["value"] for row in comparable]
    leakage = [row["private_character_leakage"] for row in comparable]
    overmask = [row["non_private_overmasking_rate"] for row in comparable]

    figure, (left, right) = plt.subplots(1, 2, figsize=(10.5, 4.2))
    colours = [SERIES["secondary"], SERIES["primary"]]

    # Bars start at zero. The two models are genuinely close on this split, and a
    # cropped axis that turned 0.0086 F1 into a towering difference would be a lie;
    # the gap that matters is in the error rates on the right.
    bars = left.bar(names, f1, color=colours, width=0.55)
    for bar, value in zip(bars, f1):
        left.annotate(f"{value:.4f}", (bar.get_x() + bar.get_width() / 2, value),
                      ha="center", va="bottom", xytext=(0, 4),
                      textcoords="offset points", fontsize=10, color=INK)
    left.set_ylim(0, 1.08)
    left.set_ylabel("Typed entity F1")
    left.set_title(f"Exact-span, correct-type F1  (+{f1[1] - f1[0]:.4f})")
    tidy(left)

    width = 0.35
    positions = range(len(names))
    error_bars_leak = right.bar([p - width / 2 for p in positions], leakage,
                                width=width, color=colours[0], label="Private character leakage")
    error_bars_over = right.bar([p + width / 2 for p in positions], overmask,
                                width=width, color=colours[1], label="Non-private overmasking")
    for group, values in ((error_bars_leak, leakage), (error_bars_over, overmask)):
        for bar, value in zip(group, values):
            right.annotate(f"{value:.4f}", (bar.get_x() + bar.get_width() / 2, value),
                           ha="center", va="bottom", xytext=(0, 3),
                           textcoords="offset points", fontsize=8.5, color=MUTED)
    right.set_xticks(list(positions), names)
    right.set_ylabel("Rate (lower is better)")
    right.set_title("Character-level error rates")
    right.set_ylim(0, max(leakage + overmask) * 1.6)
    right.legend(loc="upper center", ncol=2)
    tidy(right)

    figure.suptitle(
        "Privacy model generations on the shared held-out validation split "
        f"({head_to_head['rows']:,} rows)",
        fontsize=13, y=1.02,
    )
    figure.text(
        0.5, -0.06,
        "The earlier BiLSTM baseline has no span-level evaluation on record "
        f"(token-level binary F1 {BILSTM['val_f1']:.4f} on its own split), so it is "
        "listed in the table rather than plotted here."
        if BILSTM else "",
        ha="center", fontsize=9, color=MUTED,
    )
    save(figure, "model_progression",
         "Typed entity F1 and character-level error rates for TinyBERT INT8 and "
         "DeBERTa INT8 on the same held-out validation split.")
else:
    print("skipped: no shared head-to-head validation record")

save_table(
    progression_rows,
    ["model", "generation", "precision", "metric", "value", "precision_value",
     "recall_value", "evaluation", "comparable_to_span_metrics",
     "private_character_leakage", "non_private_overmasking_rate"],
    "model_progression",
    "Model generations with the metric each one actually has on record.",
)

## 3. Quantization: FP32 against INT8

Both runtimes were decoded by the same production decoder, so the difference is
quantization and nothing else. The dashed line is the release gate: a view fails
if it loses more than 0.02 typed entity F1.

In [ ]:
GATE_LIMIT = 0.02
quantization_rows = []

if DEBERTA_FP32 and DEBERTA_INT8:
    fp32 = DEBERTA_FP32["summary"]
    int8 = DEBERTA_INT8["summary"]
    views = [view for view in VIEW_ORDER if view in fp32 and view in int8]

    for view in views:
        quantization_rows.append({
            "view": view,
            "rows": int8[view]["rows"],
            "fp32_typed_f1": fp32[view]["typed_entity_f1"],
            "int8_typed_f1": int8[view]["typed_entity_f1"],
            "typed_f1_drop": fp32[view]["typed_entity_f1"] - int8[view]["typed_entity_f1"],
            "fp32_leakage": fp32[view]["private_character_leakage"],
            "int8_leakage": int8[view]["private_character_leakage"],
            "leakage_rise": int8[view]["private_character_leakage"]
            - fp32[view]["private_character_leakage"],
            "fp32_character_recall": fp32[view]["private_character_recall"],
            "int8_character_recall": int8[view]["private_character_recall"],
        })

    figure, (top, bottom) = plt.subplots(
        2, 1, figsize=(9.5, 7.0), gridspec_kw={"height_ratios": [2.1, 1]}
    )
    positions = range(len(views))
    width = 0.38
    fp32_values = [fp32[view]["typed_entity_f1"] for view in views]
    int8_values = [int8[view]["typed_entity_f1"] for view in views]

    fp32_bars = top.bar([p - width / 2 for p in positions], fp32_values, width=width,
                        color=SERIES["tertiary"], label="FP32 ONNX")
    int8_bars = top.bar([p + width / 2 for p in positions], int8_values, width=width,
                        color=SERIES["primary"], label="INT8 ONNX (deployed)")
    for group, values in ((fp32_bars, fp32_values), (int8_bars, int8_values)):
        for bar, value in zip(group, values):
            top.annotate(f"{value:.3f}", (bar.get_x() + bar.get_width() / 2, value),
                         ha="center", va="bottom", xytext=(0, 3),
                         textcoords="offset points", fontsize=8.5, color=INK)
    top.set_xticks(list(positions), [VIEW_TICKS[view] for view in views])
    top.set_ylabel("Typed entity F1")
    top.set_ylim(0, 1.30)
    top.set_title("Typed entity F1 by evaluation view")
    top.legend(loc="upper left", ncol=2)
    tidy(top)

    drops = [row["typed_f1_drop"] for row in quantization_rows]
    drop_bars = bottom.bar(list(positions), drops, width=0.5, color=SERIES["secondary"])
    for bar, value in zip(drop_bars, drops):
        bottom.annotate(f"{value:+.4f}", (bar.get_x() + bar.get_width() / 2, value),
                        ha="center", va="bottom", xytext=(0, 3),
                        textcoords="offset points", fontsize=9, color=INK)
    bottom.axhline(GATE_LIMIT, linestyle="--", linewidth=1.4, color=MUTED)
    bottom.annotate(f"release gate {GATE_LIMIT}", (-0.45, GATE_LIMIT),
                    ha="left", va="bottom", xytext=(0, 4), textcoords="offset points",
                    fontsize=9, color=MUTED)
    bottom.set_xticks(list(positions), [VIEW_TICKS[view] for view in views])
    bottom.set_ylabel("F1 lost to INT8")
    bottom.set_ylim(0, max(GATE_LIMIT, max(drops)) * 1.75)
    bottom.set_title("Cost of quantization, against the acceptance limit")
    tidy(bottom)

    figure.suptitle("DeBERTa-v3-xsmall: FP32 against INT8 through one decoder",
                    fontsize=13, y=0.98)
    figure.tight_layout(rect=(0, 0, 1, 0.95))
    save(figure, "deberta_fp32_vs_int8",
         "FP32 and INT8 typed entity F1 on every evaluation view, with the F1 lost "
         "to quantization against the 0.02 release gate.")

    print(f"{'view':22} {'FP32':>8} {'INT8':>8} {'drop':>9} {'gate':>6}")
    for row in quantization_rows:
        print(f"{row['view']:22} {row['fp32_typed_f1']:>8.4f} {row['int8_typed_f1']:>8.4f} "
              f"{row['typed_f1_drop']:>+9.4f} {'PASS' if row['typed_f1_drop'] < GATE_LIMIT else 'FAIL':>6}")
else:
    print("skipped: the FP32 or INT8 production-path evaluation is missing")

save_table(
    quantization_rows,
    ["view", "rows", "fp32_typed_f1", "int8_typed_f1", "typed_f1_drop",
     "fp32_leakage", "int8_leakage", "leakage_rise",
     "fp32_character_recall", "int8_character_recall"],
    "deberta_quantization",
    "FP32 against INT8 on every view, with the deltas the release gate reads.",
)

## 4. Where the deployed model is strong and where it is not

The same INT8 artifact across every view it was measured on, including the
one-shot frozen OOD final benchmark. The in-distribution views and the
out-of-distribution views are deliberately drawn together: the gap between them
is the honest statement of what this model does on speech it has never seen.

In [ ]:
split_rows = []

if DEBERTA_INT8:
    summary = dict(DEBERTA_INT8["summary"])
    if OOD_FINAL_DEBERTA:
        headline = OOD_FINAL_DEBERTA["headline"]
        metrics = OOD_FINAL_DEBERTA["metrics"]
        summary["ood_final"] = {
            "typed_entity_f1": headline["typed_entity_f1"],
            "typed_entity_precision": metrics["typed_entity"]["precision"],
            "typed_entity_recall": metrics["typed_entity"]["recall"],
            "binary_entity_f1": headline["binary_entity_f1"],
            "private_character_recall": headline["private_character_recall"],
            "private_character_leakage": headline["private_character_leakage"],
            "non_private_overmasking_rate": headline["non_private_overmasking_rate"],
            "hard_negative_false_positive_rate": headline["hard_negative_false_positive_rate"],
            "rows": OOD_FINAL_DEBERTA["fixture"]["case_count"],
        }

    views = [view for view in VIEW_ORDER + ["ood_final"] if view in summary]
    for view in views:
        entry = summary[view]
        split_rows.append({
            "view": view,
            "rows": entry["rows"],
            "typed_precision": entry["typed_entity_precision"],
            "typed_recall": entry["typed_entity_recall"],
            "typed_f1": entry["typed_entity_f1"],
            "binary_f1": entry["binary_entity_f1"],
            "private_character_recall": entry["private_character_recall"],
            "private_character_leakage": entry["private_character_leakage"],
            "non_private_overmasking_rate": entry["non_private_overmasking_rate"],
            "hard_negative_false_positive_rate": entry["hard_negative_false_positive_rate"],
        })

    figure, (left, right) = plt.subplots(1, 2, figsize=(13.5, 4.8))
    positions = range(len(views))
    width = 0.27
    triples = (
        ("Precision", [row["typed_precision"] for row in split_rows], SERIES["tertiary"]),
        ("Recall", [row["typed_recall"] for row in split_rows], SERIES["secondary"]),
        ("F1", [row["typed_f1"] for row in split_rows], SERIES["primary"]),
    )
    for index, (name, values, colour) in enumerate(triples):
        offset = (index - 1) * width
        bars = left.bar([p + offset for p in positions], values, width=width,
                        color=colour, label=name)
        if name == "F1":
            for bar, value in zip(bars, values):
                left.annotate(f"{value:.3f}", (bar.get_x() + bar.get_width() / 2, value),
                              ha="center", va="bottom", xytext=(0, 3),
                              textcoords="offset points", fontsize=8.5, color=INK)
    left.set_xticks(list(positions), [VIEW_TICKS[view] for view in views])
    left.set_ylim(0, 1.32)
    left.set_ylabel("Typed entity score")
    left.set_title("Exact-span, correct-type performance")
    left.legend(loc="upper left", ncol=3)
    tidy(left)

    recall = [row["private_character_recall"] for row in split_rows]
    bars = right.bar(list(positions), recall, width=0.55, color=SERIES["primary"])
    for bar, value in zip(bars, recall):
        right.annotate(f"{value:.3f}", (bar.get_x() + bar.get_width() / 2, value),
                       ha="center", va="bottom", xytext=(0, 3),
                       textcoords="offset points", fontsize=9, color=INK)
    right.set_xticks(list(positions), [VIEW_TICKS[view] for view in views])
    right.set_ylim(0, 1.15)
    right.set_ylabel("Private characters covered")
    right.set_title("Character recall - the privacy-relevant measure")
    right.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    tidy(right)

    figure.suptitle("Deployed DeBERTa-v3-xsmall INT8 across every evaluation view",
                    fontsize=13, y=1.01)
    save(figure, "deberta_split_performance",
         "Typed entity precision/recall/F1 and private-character recall for the "
         "deployed INT8 model on each evaluation view.")

    for row in split_rows:
        print(f"{row['view']:22} rows {row['rows']:>7,}  typed F1 {row['typed_f1']:.4f}  "
              f"char recall {row['private_character_recall']:.4f}  "
              f"leakage {row['private_character_leakage']:.4f}")
else:
    print("skipped: the INT8 production-path evaluation is missing")

save_table(
    split_rows,
    ["view", "rows", "typed_precision", "typed_recall", "typed_f1", "binary_f1",
     "private_character_recall", "private_character_leakage",
     "non_private_overmasking_rate", "hard_negative_false_positive_rate"],
    "deberta_split_performance",
    "Every metric of the deployed INT8 model on every evaluation view.",
)

## 5. Per-entity performance

Entity types, not BIO labels: 31 entity rows are readable, 63 tag rows are not,
and the entity is the unit the mask policy is written in. Measured on the
combined validation view, the same one checkpoint selection used.

Bars are ordered worst to best, and each is annotated with its gold support, so
a low F1 on three spans is not mistaken for a low F1 on three thousand.

In [ ]:
per_entity_rows = []

if DEBERTA_INT8 and "combined_validation" in DEBERTA_INT8["views"]:
    per_label = DEBERTA_INT8["views"]["combined_validation"]["per_label"]
    per_entity_rows = sorted(
        (
            {
                "entity": entity,
                "precision": entry["precision"],
                "recall": entry["recall"],
                "f1": entry["f1"],
                "support": entry["support"],
                "predicted": entry["predicted"],
                "true_positives": entry["true_positives"],
            }
            for entity, entry in per_label.items()
        ),
        key=lambda row: row["f1"],
    )

    height = max(5.0, 0.30 * len(per_entity_rows) + 1.4)
    figure, axis = plt.subplots(figsize=(9.0, height))
    labels = [row["entity"] for row in per_entity_rows]
    values = [row["f1"] for row in per_entity_rows]
    positions = range(len(labels))
    bars = axis.barh(list(positions), values, height=0.62, color=SERIES["primary"])
    for bar, row in zip(bars, per_entity_rows):
        axis.annotate(
            f"{row['f1']:.3f}   n={row['support']:,}",
            (row["f1"], bar.get_y() + bar.get_height() / 2),
            va="center", ha="left", xytext=(5, 0), textcoords="offset points",
            fontsize=8.5, color=MUTED,
        )
    axis.set_yticks(list(positions), labels)
    axis.set_xlim(0, 1.18)
    axis.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    axis.set_xlabel("Typed entity F1 (exact span, correct type)")
    axis.set_title(
        "Per-entity performance, deployed INT8 model\n"
        f"combined validation, {DEBERTA_INT8['summary']['combined_validation']['rows']:,} rows",
        loc="left",
    )
    tidy(axis, x_grid=True)
    save(figure, "per_entity_f1",
         "Typed entity F1 per entity type on the combined validation view, "
         "annotated with gold support.")

    weakest = per_entity_rows[:5]
    print("weakest entity types:")
    for row in weakest:
        print(f"  {row['entity']:20} F1 {row['f1']:.4f}  support {row['support']:,}")
else:
    print("skipped: no per-label breakdown available")

save_table(
    per_entity_rows,
    ["entity", "precision", "recall", "f1", "support", "predicted", "true_positives"],
    "per_entity_f1",
    "Per-entity precision, recall, F1 and support on the combined validation view.",
)

## 6. Out-of-distribution: the two deployment models on the same fixtures

The strongest comparison in the project. Both models were scored on the two
purpose-built OOD fixtures - 360 development cases and 360 frozen final cases,
byte-identical files, the same metric code, both in their model-only form. The
frozen set was spent exactly once per model.

This is where a model trained on generated corpora meets speech-shaped input it
has never seen, and it separates the two generations far more sharply than the
in-distribution split does.

In [ ]:
def tinybert_ood(summary, stage: str = "int8_model"):
    if not summary or stage not in summary.get("stages", {}):
        return None
    overall = summary["stages"][stage]["overall"]
    return {
        "typed_f1": overall["typed_entity"]["f1"],
        "typed_precision": overall["typed_entity"]["precision"],
        "typed_recall": overall["typed_entity"]["recall"],
        "character_recall": overall["privacy_character"]["private_character_recall"],
        "character_leakage": overall["privacy_character"]["private_character_leakage_rate"],
        "overmasking": overall["privacy_character"]["non_private_character_overmasking_rate"],
        "hard_negative_fp": overall["case_level"]["hard_negative_false_positive_case_rate"],
        "rows": overall["case_count"],
    }


def deberta_ood(view: str):
    if view == "ood_final":
        if not OOD_FINAL_DEBERTA:
            return None
        metrics, headline = OOD_FINAL_DEBERTA["metrics"], OOD_FINAL_DEBERTA["headline"]
        return {
            "typed_f1": headline["typed_entity_f1"],
            "typed_precision": metrics["typed_entity"]["precision"],
            "typed_recall": metrics["typed_entity"]["recall"],
            "character_recall": headline["private_character_recall"],
            "character_leakage": headline["private_character_leakage"],
            "overmasking": headline["non_private_overmasking_rate"],
            "hard_negative_fp": headline["hard_negative_false_positive_rate"],
            "rows": OOD_FINAL_DEBERTA["fixture"]["case_count"],
        }
    if not DEBERTA_INT8 or view not in DEBERTA_INT8["summary"]:
        return None
    entry = DEBERTA_INT8["summary"][view]
    return {
        "typed_f1": entry["typed_entity_f1"],
        "typed_precision": entry["typed_entity_precision"],
        "typed_recall": entry["typed_entity_recall"],
        "character_recall": entry["private_character_recall"],
        "character_leakage": entry["private_character_leakage"],
        "overmasking": entry["non_private_overmasking_rate"],
        "hard_negative_fp": entry["hard_negative_false_positive_rate"],
        "rows": entry["rows"],
    }


ood_rows = []
pairs = [
    ("OOD dev", tinybert_ood(TINYBERT_OOD_DEV), deberta_ood("ood_dev")),
    ("OOD final (frozen)", tinybert_ood(TINYBERT_OOD_FINAL), deberta_ood("ood_final")),
]
usable = [(name, old, new) for name, old, new in pairs if old and new]

for name, old, new in usable:
    for model, entry in (("TinyBERT-4 INT8", old), ("DeBERTa-v3-xsmall INT8", new)):
        ood_rows.append({"fixture": name, "model": model, **entry})

if usable:
    figure, axes = plt.subplots(1, 3, figsize=(13.0, 4.4))
    panels = (
        ("typed_f1", "Typed entity F1", "higher is better"),
        ("character_recall", "Private character recall", "higher is better"),
        ("hard_negative_fp", "Hard-negative false-positive rate", "lower is better"),
    )
    names = [name for name, _, _ in usable]
    positions = range(len(names))
    width = 0.36

    for axis, (key, title, direction) in zip(axes, panels):
        old_values = [old[key] for _, old, _ in usable]
        new_values = [new[key] for _, _, new in usable]
        old_bars = axis.bar([p - width / 2 for p in positions], old_values, width=width,
                            color=SERIES["secondary"], label="TinyBERT-4 INT8 (v1)")
        new_bars = axis.bar([p + width / 2 for p in positions], new_values, width=width,
                            color=SERIES["primary"], label="DeBERTa-v3-xsmall INT8 (v2)")
        for group, values in ((old_bars, old_values), (new_bars, new_values)):
            for bar, value in zip(group, values):
                axis.annotate(f"{value:.3f}", (bar.get_x() + bar.get_width() / 2, value),
                              ha="center", va="bottom", xytext=(0, 3),
                              textcoords="offset points", fontsize=8.5, color=INK)
        axis.set_xticks(list(positions), names)
        axis.set_ylim(0, 1.05)
        axis.set_title(f"{title}\n({direction})", fontsize=11)
        tidy(axis)
    axes[0].set_ylabel("Score")
    axes[0].legend(loc="upper left")

    figure.suptitle(
        "Out-of-distribution behaviour on identical 360-case fixtures, model-only paths",
        fontsize=13, y=1.03,
    )
    save(figure, "ood_model_comparison",
         "TinyBERT INT8 against DeBERTa INT8 on the OOD development and frozen "
         "final fixtures, model-only in both cases.")

    for row in ood_rows:
        print(f"{row['fixture']:20} {row['model']:24} typed F1 {row['typed_f1']:.4f}  "
              f"char recall {row['character_recall']:.4f}  hard-neg FP {row['hard_negative_fp']:.4f}")
else:
    print("skipped: OOD results are not available for both models")

save_table(
    ood_rows,
    ["fixture", "model", "rows", "typed_f1", "typed_precision", "typed_recall",
     "character_recall", "character_leakage", "overmasking", "hard_negative_fp"],
    "ood_model_comparison",
    "Both deployment models on the OOD development and frozen final fixtures.",
)

## 7. Deployment size

What has to fit on the device. Sizes are read from the artifacts themselves
where they exist, and from the recorded metrics where the file is a local build
artifact rather than a committed one.

In [ ]:
size_rows = []


def record_size(model: str, precision: str, path: Path | None, fallback_bytes=None, note: str = ""):
    if path is not None and Path(path).is_file():
        size = Path(path).stat().st_size
        source = Path(path).relative_to(REPO_ROOT).as_posix()
    elif fallback_bytes:
        size, source = int(fallback_bytes), "recorded in saved metrics"
    else:
        return
    size_rows.append({
        "model": model, "precision": precision, "bytes": size,
        "mib": size / 1024 ** 2, "source": source, "note": note,
    })


record_size("DeBERTa-v3-xsmall", "FP32 ONNX",
            MODEL_V2 / "artifacts" / "model_v2" / "models" / "fp32" / "model_fp32.onnx",
            (ACCEPTED or {}).get("model", {}).get("fp32_bytes"),
            "reference graph, not deployed")
record_size("DeBERTa-v3-xsmall", "INT8 ONNX",
            REPO_ROOT / "FullPipeline_V3" / "models" / "deberta_pii" / "model_int8.onnx",
            (ACCEPTED or {}).get("model", {}).get("int8_bytes"),
            "deployed")
if TINYBERT_IID:
    record_size("TinyBERT-4", "FP32 ONNX", None,
                TINYBERT_IID["onnx_fp32"]["model_size_mib"] * 1024 ** 2, "previous generation")
    record_size("TinyBERT-4", "INT8 ONNX",
                REPO_ROOT / "tinybert_pii_rpi" / "model_int8.onnx",
                TINYBERT_IID["onnx_int8"]["model_size_mib"] * 1024 ** 2,
                "previous deployment")
record_size("BiLSTM baseline", "FP32 PyTorch",
            REPO_ROOT / "model" / "artifacts" / "pii_masking_bilstm.pt",
            None, "first baseline")

if size_rows:
    ordered = sorted(size_rows, key=lambda row: row["mib"])
    labels = [f"{row['model']}\n{row['precision']}" for row in ordered]
    values = [row["mib"] for row in ordered]
    # One highlight, not a palette: the row labels already carry identity, so
    # colour is spent on the single thing the reader is looking for.
    colours = [
        SERIES["primary"] if row["note"] == "deployed" else "#b8b7b2" for row in ordered
    ]
    figure, axis = plt.subplots(figsize=(9.0, 4.4))
    bars = axis.barh(range(len(labels)), values, height=0.6, color=colours)
    for bar, row in zip(bars, ordered):
        axis.annotate(f"{row['mib']:.1f} MiB" + ("   deployed" if row["note"] == "deployed" else ""),
                      (row["mib"], bar.get_y() + bar.get_height() / 2),
                      va="center", ha="left", xytext=(6, 0), textcoords="offset points",
                      fontsize=9.5, color=INK if row["note"] == "deployed" else MUTED)
    axis.set_yticks(range(len(labels)), labels)
    axis.set_xlim(0, max(values) * 1.32)
    axis.set_xlabel("Artifact size on disk (MiB)")
    axis.set_title("What has to ship to the device", loc="left")
    tidy(axis, x_grid=True)
    save(figure, "model_size_comparison",
         "On-disk size of every privacy-model artifact, deployed one highlighted.")

    for row in ordered:
        print(f"{row['model']:20} {row['precision']:14} {row['mib']:>8.2f} MiB   {row['note']}")
else:
    print("skipped: no artifact sizes available")

save_table(size_rows, ["model", "precision", "bytes", "mib", "source", "note"],
           "model_size_comparison", "On-disk size of each privacy-model artifact.")

## 8. Reference latency by input length

From the workstation preflight benchmark, single-threaded and controlled. It
measures the *architecture* at four input lengths; the head was untrained, so
these numbers say nothing about quality and everything about cost.

This is a development-machine measurement. Raspberry Pi latency comes from
section 9.

In [ ]:
latency_rows = []

if PREFLIGHT and "one_thread_controlled" in PREFLIGHT:
    controlled = PREFLIGHT["one_thread_controlled"]
    models = controlled["models"]
    display = {
        "deberta_fp32": ("DeBERTa-v3-xsmall FP32", SERIES["tertiary"], "--"),
        "deberta_int8": ("DeBERTa-v3-xsmall INT8", SERIES["primary"], "-"),
        "tinybert_int8": ("TinyBERT-4 INT8", SERIES["secondary"], "-"),
    }
    lengths = sorted({int(key) for entry in models.values() for key in entry})

    figure, axis = plt.subplots(figsize=(8.6, 4.8))
    for key, (name, colour, style) in display.items():
        if key not in models:
            continue
        values = [models[key][str(length)]["median_ms"] for length in lengths
                  if str(length) in models[key]]
        present = [length for length in lengths if str(length) in models[key]]
        axis.plot(present, values, marker="o", color=colour, linestyle=style, label=name)
        axis.annotate(f"{values[-1]:.0f} ms", (present[-1], values[-1]),
                      xytext=(6, 0), textcoords="offset points",
                      va="center", fontsize=9.5, color=colour)
        for length in present:
            entry = models[key][str(length)]
            latency_rows.append({
                "model": name, "tokens": length, "repetitions": entry["repetitions"],
                "mean_ms": entry["mean_ms"], "median_ms": entry["median_ms"],
                "p95_ms": entry["p95_ms"], "min_ms": entry["min_ms"], "max_ms": entry["max_ms"],
                "threads": controlled["intra_op_num_threads"],
                "host": PREFLIGHT["host"]["processor"],
            })

    axis.set_xticks(lengths)
    axis.set_xlabel("Input length (tokens)")
    axis.set_ylabel("Median single-inference latency (ms)")
    axis.set_xlim(min(lengths) - 20, max(lengths) + 90)
    axis.set_ylim(0, None)
    axis.set_title(
        "Single-threaded inference cost against input length\n"
        f"{PREFLIGHT['host']['processor']}, {controlled['repetitions']} repetitions "
        f"after {controlled['warmups']} warm-ups",
        loc="left",
    )
    axis.legend(loc="upper left")
    tidy(axis)
    save(figure, "latency_by_input_length",
         "Median single-threaded inference latency against input length for all "
         "three deployment candidates, on the development workstation.")
else:
    print("skipped: no preflight latency benchmark available")

save_table(
    latency_rows,
    ["model", "tokens", "repetitions", "mean_ms", "median_ms", "p95_ms",
     "min_ms", "max_ms", "threads", "host"],
    "latency_by_input_length",
    "Single-threaded latency at four input lengths on the development workstation.",
)

## 9. CPU deployment benchmarks

Whatever `Full_Pipeline_CPU_Benchmark.ipynb` has produced. One run gives the
latency, stage-breakdown and memory figures. Two runs on different devices add
the comparison and the prediction-parity check.

Nothing here fails when the results do not exist yet.

In [ ]:
RUNS = []
for directory in benchmark_runs:
    config = load_json(directory / "benchmark_config.json", f"{directory.name} config")
    system = load_json(directory / "system_info.json", f"{directory.name} system")
    if not config or not system:
        continue
    RUNS.append({
        "directory": directory,
        "run_id": directory.name,
        "config": config,
        "system": system,
        "latency": load_json(directory / "deberta_latency_summary.json", f"{directory.name} latency"),
        "pipeline": load_json(directory / "pipeline_latency_summary.json", f"{directory.name} pipeline"),
        "evaluation": load_json(directory / "evaluation_metrics.json", f"{directory.name} evaluation"),
        "memory": load_json(directory / "memory_metrics.json", f"{directory.name} memory"),
        "artifacts": load_json(directory / "artifact_manifest.json", f"{directory.name} artifacts"),
    })


def device_name(run) -> str:
    system = run["system"]
    board = system.get("board_model")
    if board:
        return board.replace("Raspberry Pi", "Pi").strip()
    label = run["config"].get("device_label")
    if label:
        return label
    return f"{system.get('os', '?')} {system.get('architecture', '?')}"


if not RUNS:
    print("No CPU benchmark runs found yet.")
    print()
    print("Deployment figures become available once Full_Pipeline_CPU_Benchmark.ipynb")
    print("has been run and its output folder committed. Run it here for a workstation")
    print("baseline, and on the Raspberry Pi for the device comparison, then rerun this")
    print("notebook - nothing needs editing.")
else:
    print(f"{'run':46} {'device':26} {'mode':6} {'rows':>6}")
    print("-" * 88)
    for run in RUNS:
        print(f"{run['run_id']:46} {device_name(run):26} "
              f"{run['config']['mode']:6} {run['config']['rows_scored']:>6,}")

run_rows = [
    {
        "run_id": run["run_id"],
        "device": device_name(run),
        "os": run["system"].get("os"),
        "architecture": run["system"].get("architecture"),
        "cpu": run["system"].get("cpu_model"),
        "logical_cpus": run["system"].get("logical_cpus"),
        "total_ram_gib": (run["system"].get("total_ram_bytes") or 0) / 1024 ** 3,
        "mode": run["config"]["mode"],
        "rows_scored": run["config"]["rows_scored"],
        "int8_sha256": (run["artifacts"] or {}).get("files", {})
        .get("model_int8.onnx", {}).get("sha256"),
        "typed_f1": (run["evaluation"] or {}).get("production_path", {})
        .get("typed_entity", {}).get("f1"),
        "mean_latency_ms": (run["latency"] or {}).get("end_to_end_masking_call", {}).get("mean_ms"),
        "p95_latency_ms": (run["latency"] or {}).get("end_to_end_masking_call", {}).get("p95_ms"),
        "throughput_rows_per_second": (run["latency"] or {}).get("throughput_rows_per_second"),
        "peak_rss_mib": ((run["memory"] or {}).get("process_peak_rss_bytes") or 0) / 1024 ** 2,
    }
    for run in RUNS
]
save_table(
    run_rows,
    ["run_id", "device", "os", "architecture", "cpu", "logical_cpus", "total_ram_gib",
     "mode", "rows_scored", "int8_sha256", "typed_f1", "mean_latency_ms",
     "p95_latency_ms", "throughput_rows_per_second", "peak_rss_mib"],
    "cpu_benchmark_runs",
    "Every CPU benchmark run discovered, with its device and headline numbers.",
)

In [ ]:
# --- latency distribution -----------------------------------------------------
def read_latency_csv(directory: Path):
    path = directory / "deberta_latency.csv"
    if not path.is_file():
        return []
    with path.open(encoding="utf-8", newline="") as stream:
        return [
            {"latency_ms": float(row["latency_ms"]), "characters": int(row["characters"])}
            for row in csv.DictReader(stream)
        ]


plotted = [(run, read_latency_csv(run["directory"])) for run in RUNS]
plotted = [(run, samples) for run, samples in plotted if samples]

if plotted:
    figure, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.6))
    colours = [SERIES["primary"], SERIES["secondary"], SERIES["tertiary"]]

    for index, (run, samples) in enumerate(plotted[:3]):
        values = sorted(sample["latency_ms"] for sample in samples)
        colour = colours[index]
        left.hist(values, bins=40, color=colour, alpha=0.55 if index else 0.8,
                  label=f"{device_name(run)} (n={len(values):,})")
        # Empirical CDF: percentiles are read off it directly, no binning artefact.
        right.plot(values, [rank / len(values) for rank in range(1, len(values) + 1)],
                   color=colour, label=device_name(run))
        summary = (run["latency"] or {}).get("end_to_end_masking_call", {})
        for fraction, key in ((0.50, "p50_ms"), (0.95, "p95_ms"), (0.99, "p99_ms")):
            if key in summary:
                right.plot([summary[key]], [fraction], marker="o", color=colour)
                right.annotate(f"P{int(fraction * 100)} {summary[key]:.0f} ms",
                               (summary[key], fraction), xytext=(6, -2),
                               textcoords="offset points", fontsize=8.5, color=colour)

    left.set_xlabel("Masking latency per row (ms)")
    left.set_ylabel("Rows")
    left.set_title("Latency distribution")
    left.legend(loc="upper right")
    tidy(left)

    right.set_xlabel("Masking latency per row (ms)")
    right.set_ylabel("Share of rows at or below")
    right.set_ylim(0, 1.02)
    right.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    right.set_title("Cumulative distribution, with P50 / P95 / P99")
    if len(plotted) > 1:
        right.legend(loc="lower right")
    tidy(right)

    figure.suptitle("End-to-end privacy masking latency on CPU", fontsize=13, y=1.02)
    save(figure, "cpu_latency_distribution",
         "Per-row masking latency distribution and cumulative distribution for "
         "every benchmarked device.")

    percentile_rows = [
        {"run_id": run["run_id"], "device": device_name(run),
         **{key: value for key, value in
            (run["latency"] or {}).get("end_to_end_masking_call", {}).items()}}
        for run, _ in plotted
    ]
    save_table(
        percentile_rows,
        ["run_id", "device", "count", "mean_ms", "std_ms", "min_ms", "p50_ms",
         "p90_ms", "p95_ms", "p99_ms", "max_ms", "total_ms"],
        "cpu_latency_percentiles",
        "Latency percentiles of the full masking call, per device.",
    )
else:
    print("no per-row latency samples recorded yet")

In [ ]:
# --- stage breakdown ----------------------------------------------------------
if RUNS:
    run = RUNS[-1]
    stage_values = {}
    latency = run["latency"] or {}
    for name, key in (("Neural decode (ONNX + span consolidation)", "neural_decode"),
                      ("Structured regex layer", "structured"),
                      ("Span merge and mask policy", "span_merge")):
        entry = latency.get("stages", {}).get(key)
        if entry and entry.get("count"):
            stage_values[name] = entry["mean_ms"]

    pipeline = (run["pipeline"] or {}).get("stages", {})
    pipeline_labels = {
        "vad_silero": "Silero VAD (per 3 s of audio)",
        "stt_canary": "Canary STT (per 3 s of audio)",
        "transcript_merge": "Transcript merge",
        "wake_word_normalisation": "Wake-word normalisation",
        "inactivity_buffer": "Inactivity buffer",
        "privacy_masking": "Privacy masking",
        "digit_normalisation": "Spoken-digit rendering",
    }
    pipeline_values, pipeline_missing = {}, []
    for key, label in pipeline_labels.items():
        entry = pipeline.get(key)
        if not entry:
            continue
        if entry.get("available") and entry.get("count"):
            pipeline_values[label] = entry["mean_ms"]
        else:
            pipeline_missing.append(label)

    if stage_values or pipeline_values:
        panels = int(bool(stage_values)) + int(bool(pipeline_values))
        figure, axes = plt.subplots(panels, 1, figsize=(9.6, 3.0 * panels + 1.0))
        axes = [axes] if panels == 1 else list(axes)

        for axis, (title, values) in zip(
            axes,
            [item for item in (("Inside one masking call", stage_values),
                               ("Across the assembled pipeline", pipeline_values)) if item[1]],
        ):
            names = list(values)
            magnitudes = [values[name] for name in names]
            bars = axis.barh(range(len(names)), magnitudes, height=0.6,
                             color=SERIES["primary"])
            for bar, value in zip(bars, magnitudes):
                axis.annotate(f"{value:.2f} ms", (value, bar.get_y() + bar.get_height() / 2),
                              va="center", ha="left", xytext=(6, 0),
                              textcoords="offset points", fontsize=9.5, color=INK)
            axis.set_yticks(range(len(names)), names)
            # Linear only. A log axis would make the sub-millisecond text stages
            # look comparable to the neural decode, which is the opposite of the
            # point; the annotations carry the small values instead.
            axis.set_xlim(0, max(magnitudes) * 1.30)
            axis.set_xlabel("Mean latency (ms)")
            axis.set_title(title, loc="left")
            axis.invert_yaxis()
            tidy(axis, x_grid=True)

        subtitle = f"{device_name(run)} - {run['config']['mode']} run"
        if pipeline_missing:
            subtitle += f"\nnot measurable on this device: {', '.join(pipeline_missing)}"
        figure.suptitle(f"Where the time goes\n{subtitle}", fontsize=12.5, y=1.0)
        figure.tight_layout()
        save(figure, "pipeline_latency_breakdown",
             "Mean latency of each stage inside a masking call and across the "
             "assembled pipeline, on the most recent benchmarked device.")

        save_table(
            [{"run_id": run["run_id"], "device": device_name(run),
              "scope": scope, "stage": name, "mean_ms": value}
             for scope, values in (("masking_call", stage_values),
                                   ("pipeline", pipeline_values))
             for name, value in values.items()],
            ["run_id", "device", "scope", "stage", "mean_ms"],
            "pipeline_latency_breakdown",
            "Mean latency per stage for the most recent benchmarked device.",
        )
    else:
        print("no stage latency recorded in the most recent run")
else:
    print("no CPU benchmark runs to break down")

In [ ]:
# --- memory and throughput ----------------------------------------------------
memory_runs = [run for run in RUNS if (run["memory"] or {}).get("process_peak_rss_bytes")]

if memory_runs:
    figure, (left, right) = plt.subplots(1, 2, figsize=(11.5, 4.3))
    names = [device_name(run) for run in memory_runs]
    positions = range(len(names))

    peak = [(run["memory"]["process_peak_rss_bytes"]) / 1024 ** 2 for run in memory_runs]
    stack = [
        (run["memory"].get("privacy_stack_resident_bytes") or 0) / 1024 ** 2
        for run in memory_runs
    ]
    width = 0.36
    peak_bars = left.bar([p - width / 2 for p in positions], peak, width=width,
                         color=SERIES["primary"], label="Process peak RSS")
    stack_bars = left.bar([p + width / 2 for p in positions], stack, width=width,
                          color=SERIES["tertiary"], label="Privacy stack resident")
    for group, values in ((peak_bars, peak), (stack_bars, stack)):
        for bar, value in zip(group, values):
            left.annotate(f"{value:,.0f}", (bar.get_x() + bar.get_width() / 2, value),
                          ha="center", va="bottom", xytext=(0, 3),
                          textcoords="offset points", fontsize=9, color=INK)
    for index, run in enumerate(memory_runs):
        total = run["system"].get("total_ram_bytes")
        if total:
            left.annotate(f"{total / 1024 ** 3:.1f} GiB RAM", (index, 0),
                          ha="center", va="top", xytext=(0, -18),
                          textcoords="offset points", fontsize=8.5, color=MUTED)
    left.set_xticks(list(positions), names)
    left.set_ylabel("MiB")
    left.set_title("Memory footprint")
    left.legend(loc="upper left")
    tidy(left)

    throughput = [
        (run["latency"] or {}).get("throughput_rows_per_second") or 0 for run in memory_runs
    ]
    bars = right.bar(list(positions), throughput, width=0.5, color=SERIES["primary"])
    for bar, value in zip(bars, throughput):
        right.annotate(f"{value:.2f}", (bar.get_x() + bar.get_width() / 2, value),
                       ha="center", va="bottom", xytext=(0, 3),
                       textcoords="offset points", fontsize=9.5, color=INK)
    right.set_xticks(list(positions), names)
    right.set_ylabel("Rows masked per second")
    right.set_title("Throughput")
    tidy(right)

    figure.suptitle("Deployment resource use", fontsize=13, y=1.02)
    save(figure, "cpu_memory_and_throughput",
         "Peak process RSS, resident privacy-stack cost and masking throughput "
         "for every benchmarked device.")
else:
    print("no memory metrics recorded yet")

In [ ]:
# --- cross-device parity ------------------------------------------------------
def read_parity(directory: Path) -> dict[str, str]:
    path = directory / "prediction_parity_data.jsonl"
    if not path.is_file():
        return {}
    records = {}
    with path.open(encoding="utf-8") as stream:
        for line in stream:
            if line.strip():
                record = json.loads(line)
                records[record["row_id"]] = record["production_digest"]
    return records


parity_rows = []
comparable = [run for run in RUNS if (run["evaluation"] or {}).get("production_path")]

if len(comparable) >= 2:
    reference = comparable[0]
    reference_digests = read_parity(reference["directory"])
    for other in comparable[1:]:
        other_digests = read_parity(other["directory"])
        shared = sorted(set(reference_digests) & set(other_digests))
        agree = sum(1 for row_id in shared if reference_digests[row_id] == other_digests[row_id])
        left_metrics = reference["evaluation"]["production_path"]
        right_metrics = other["evaluation"]["production_path"]
        parity_rows.append({
            "reference_run": reference["run_id"],
            "reference_device": device_name(reference),
            "comparison_run": other["run_id"],
            "comparison_device": device_name(other),
            "shared_rows": len(shared),
            "identical_predictions": agree,
            "agreement": agree / len(shared) if shared else None,
            "reference_typed_f1": left_metrics["typed_entity"]["f1"],
            "comparison_typed_f1": right_metrics["typed_entity"]["f1"],
            "typed_f1_delta": right_metrics["typed_entity"]["f1"] - left_metrics["typed_entity"]["f1"],
            "character_recall_delta": right_metrics["private_character_recall"]
            - left_metrics["private_character_recall"],
            "same_model_hash": (reference["artifacts"] or {}).get("files", {})
            .get("model_int8.onnx", {}).get("sha256")
            == (other["artifacts"] or {}).get("files", {})
            .get("model_int8.onnx", {}).get("sha256"),
        })

    figure, (left, right) = plt.subplots(1, 2, figsize=(11.5, 4.3))
    labels = [f"{row['reference_device']}\nvs {row['comparison_device']}" for row in parity_rows]
    agreement = [row["agreement"] or 0 for row in parity_rows]
    positions = range(len(labels))
    bars = left.bar(list(positions), agreement, width=0.45, color=SERIES["primary"])
    for bar, row in zip(bars, parity_rows):
        left.annotate(f"{row['agreement']:.4%}\n{row['identical_predictions']:,}/{row['shared_rows']:,}",
                      (bar.get_x() + bar.get_width() / 2, row["agreement"]),
                      ha="center", va="bottom", xytext=(0, 4),
                      textcoords="offset points", fontsize=9, color=INK)
    left.set_xticks(list(positions), labels)
    left.set_ylim(0, 1.12)
    left.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    left.set_ylabel("Rows with identical predicted spans")
    left.set_title("Prediction agreement across devices")
    tidy(left)

    deltas = [row["typed_f1_delta"] for row in parity_rows]
    bars = right.bar(list(positions), deltas, width=0.45,
                     color=[SERIES["primary"] if abs(value) < 1e-9 else SERIES["secondary"]
                            for value in deltas])
    for bar, value in zip(bars, deltas):
        right.annotate(f"{value:+.6f}", (bar.get_x() + bar.get_width() / 2, value),
                       ha="center", va="bottom", xytext=(0, 4),
                       textcoords="offset points", fontsize=9, color=INK)
    right.axhline(0, color=MUTED, linewidth=1.0)
    right.set_xticks(list(positions), labels)
    right.set_ylabel("Typed entity F1 difference")
    right.set_title("Metric difference across devices")
    limit = max(0.001, max(abs(value) for value in deltas) * 1.6)
    right.set_ylim(-limit, limit)
    tidy(right)

    figure.suptitle("Same INT8 model, same rows, different CPUs", fontsize=13, y=1.02)
    save(figure, "deployment_parity",
         "Prediction agreement and typed-F1 difference between benchmarked devices "
         "running the identical INT8 artifact.")

    for row in parity_rows:
        print(f"{row['reference_device']} vs {row['comparison_device']}")
        print(f"  shared rows          {row['shared_rows']:,}")
        print(f"  identical predictions {row['identical_predictions']:,} "
              f"({row['agreement']:.4%})")
        print(f"  typed F1 delta        {row['typed_f1_delta']:+.6f}")
        print(f"  same model hash       {row['same_model_hash']}")
else:
    print(f"parity needs two benchmark runs; {len(comparable)} available.")
    print("Run Full_Pipeline_CPU_Benchmark.ipynb on the Raspberry Pi, commit the")
    print("output folder, pull it here and rerun this notebook.")

save_table(
    parity_rows,
    ["reference_run", "reference_device", "comparison_run", "comparison_device",
     "shared_rows", "identical_predictions", "agreement", "reference_typed_f1",
     "comparison_typed_f1", "typed_f1_delta", "character_recall_delta", "same_model_hash"],
    "deployment_parity",
    "Cross-device prediction agreement and metric deltas.",
)

## 10. Index and overview

In [ ]:
figures = [item for item in GENERATED if item["kind"] == "figure"]
tables = [item for item in GENERATED if item["kind"] == "table"]

index = {
    "generated_from": "Research_Paper_Figures.ipynb",
    "repository_root": REPO_ROOT.name,
    "figures": figures,
    "tables": tables,
    "cpu_benchmark_runs": [run["run_id"] for run in RUNS],
    "unavailable_inputs": UNAVAILABLE,
}
index_path = SUMMARIES / "figure_index.json"
with index_path.open("w", encoding="utf-8", newline="\n") as stream:
    stream.write(json.dumps(index, indent=2) + "\n")

lines = [
    "# Research results overview",
    "",
    "Generated by `Research_Paper_Figures.ipynb`. Every number here comes from a "
    "saved evaluation artifact in this repository; nothing was recomputed.",
    "",
    f"- Figures: **{len(figures)}** (PNG at 300 dpi and PDF, in `research_results/figures/`)",
    f"- Tables: **{len(tables)}** (CSV, in `research_results/tables/`)",
    f"- CPU benchmark runs discovered: **{len(RUNS)}**",
    "",
]

if split_rows:
    deployed = {row["view"]: row for row in split_rows}
    lines += ["## Deployed model headline", "",
              "| View | Rows | Typed F1 | Character recall | Leakage |", "|---|---:|---:|---:|---:|"]
    for view, row in deployed.items():
        lines.append(
            f"| {view} | {row['rows']:,} | {row['typed_f1']:.4f} | "
            f"{row['private_character_recall']:.4f} | {row['private_character_leakage']:.4f} |"
        )
    lines.append("")

if quantization_rows:
    worst = max(quantization_rows, key=lambda row: row["typed_f1_drop"])
    lines += [
        "## Quantization",
        "",
        f"INT8 costs at most **{worst['typed_f1_drop']:.4f}** typed entity F1 "
        f"(on `{worst['view']}`), against a release gate of {GATE_LIMIT}. "
        "Every view passed.",
        "",
    ]

if ood_rows:
    lines += ["## Out of distribution", "",
              "| Fixture | Model | Typed F1 | Character recall | Hard-negative FP |",
              "|---|---|---:|---:|---:|"]
    for row in ood_rows:
        lines.append(
            f"| {row['fixture']} | {row['model']} | {row['typed_f1']:.4f} | "
            f"{row['character_recall']:.4f} | {row['hard_negative_fp']:.4f} |"
        )
    lines.append("")

if run_rows:
    lines += ["## Deployment benchmarks", "",
              "| Device | Mode | Rows | Typed F1 | Mean latency | Peak RSS |",
              "|---|---|---:|---:|---:|---:|"]
    for row in run_rows:
        lines.append(
            f"| {row['device']} | {row['mode']} | {row['rows_scored']:,} | "
            f"{row['typed_f1']:.4f} | {row['mean_latency_ms']:.1f} ms | "
            f"{row['peak_rss_mib']:,.0f} MiB |"
        )
    lines.append("")
else:
    lines += [
        "## Deployment benchmarks",
        "",
        "None yet. Run `Full_Pipeline_CPU_Benchmark.ipynb` on the workstation and on "
        "the Raspberry Pi, commit the folders it writes, then rerun this notebook.",
        "",
    ]

lines += ["## Figures", ""]
lines += [f"- `{item['name']}` - {item['caption']}" for item in figures] or ["- none"]
lines += ["", "## Tables", ""]
lines += [f"- `{item['name']}.csv` ({item['rows']} rows) - {item['caption']}"
          for item in tables] or ["- none"]

if UNAVAILABLE:
    lines += ["", "## Inputs not found", ""]
    lines += [f"- {message}" for message in UNAVAILABLE]
lines.append("")

overview_path = SUMMARIES / "results_overview.md"
overview_path.write_text("\n".join(lines), encoding="utf-8", newline="\n")

print(f"{len(figures)} figures, {len(tables)} tables")
print(f"wrote {index_path.relative_to(REPO_ROOT).as_posix()}")
print(f"wrote {overview_path.relative_to(REPO_ROOT).as_posix()}")
if UNAVAILABLE:
    print("\ninputs not found:")
    for message in UNAVAILABLE:
        print(" ", message)

---

## Done

`research_results/figures/` holds a PNG and a PDF of every plot; drop either
straight into the report. `research_results/tables/` holds the numbers behind
them, so a figure can be redrawn or a value quoted without reopening this
notebook. `research_results/summaries/results_overview.md` is the written
digest.

When the Raspberry Pi benchmark lands, pull it and press Run All again. Section
9 picks it up, and the device-comparison and parity figures appear on their own.